# 04 - Scoring Pipeline

Note: the plan named this `notebooks/03_scoring_pipeline.ipynb`, but `03`
is already taken by `03_train_model.ipynb` -- named this `04_` instead,
same reasoning as the earlier `02_train_model` -> `03_train_model` rename:
keep the notebook sequence in order rather than overwrite an existing
one.

In [1]:
import sys

sys.path.insert(0, "..")

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from app.core.config import HIGH_RISK_THRESHOLD, MEDIUM_RISK_THRESHOLD, MODEL_PATH
from app.services.feature_pipeline import build_features
from app.services.model import align_features, get_model

## `score_customers()` -- the batch-scoring mental model

This is what runs daily in production, and it's deliberately *not* a
copy of the training notebook with a few lines changed -- it's the same
`build_features()` from Day 2, called on data with no `Churn` column,
feeding `model.predict_proba()` instead of `model.fit()`. Reusing
`build_features()` exactly, rather than re-deriving the feature logic
here, is the whole point: if training and scoring features drift apart
even slightly, the model silently degrades -- it won't error, it'll just
quietly score on a feature space it was never trained for.

`risk_tier` buckets `churn_probability` using `HIGH_RISK_THRESHOLD` (0.7)
and `MEDIUM_RISK_THRESHOLD` (0.4) -- both were set up in `.env` back on
Day 1 and haven't been used until now. Note this is a *separate* decision
from the 0.780 top-15%-riskiest cutoff chosen in Day 3's evaluation: that
threshold answers "who does the retention campaign contact," a single
yes/no cutoff sized to a budget; `risk_tier` is a three-way bucketing for
reporting/triage, not a budget-constrained action list. Both read off the
same underlying probability, for different purposes.

## Risk tiering: a plain function, first

A raw probability isn't a decision -- it's an input to one. This is the
same pattern Duolingo uses: an ML score feeds a rules layer that decides
who gets a streak-save nudge. Here, that rules layer decides who gets a
retention offer.

`assign_risk_tier()` is written as a standalone function of a single
probability -- not buried inline inside `score_customers()` -- so it's
directly testable on its own. It lives in `app/services/model.py` rather
than being defined here, since `notebooks/05_scoring_chain.ipynb` needs
the exact same tier logic too -- one source of truth for the thresholds
instead of two copies that could quietly drift apart.

In [2]:
from app.services.model import assign_risk_tier

# Sanity-check the boundaries before wiring it into anything else.
assert assign_risk_tier(0.75) == "high"
assert assign_risk_tier(0.70) == "high"       # boundary is inclusive
assert assign_risk_tier(0.69999) == "medium"
assert assign_risk_tier(0.40) == "medium"     # boundary is inclusive
assert assign_risk_tier(0.39999) == "low"
assert assign_risk_tier(0.0) == "low"
print("assign_risk_tier: all boundary checks passed")

assign_risk_tier: all boundary checks passed


In [3]:
def score_customers(df: pd.DataFrame) -> pd.DataFrame:
    """Score a batch of active customers.

    df must contain the raw Telco columns plus customerID, and must NOT
    contain Churn -- this is what real scoring input looks like: customers
    whose outcome isn't known yet, which is the entire reason to be
    scoring them.

    Returns a DataFrame with customerID, churn_probability, risk_tier.
    """
    if "Churn" in df.columns:
        raise ValueError(
            "score_customers() expects unlabeled data (no Churn column) -- "
            "this simulates real scoring input for active customers."
        )

    customer_ids = df["customerID"]

    features = build_features(df)  # exact same pipeline as training (Day 2)
    artifact = get_model(path=Path("..") / MODEL_PATH)

    # Training/serving skew guard: make drift visible instead of silent.
    # A missing column usually means this batch just didn't contain some
    # category the model was trained on (common in small batches, see the
    # single-customer example below) -- align_features() fills it with 0,
    # which is correct. An *extra* column would mean a genuinely new
    # category showed up that the model has never seen and can't use --
    # that's worth knowing about, not silently dropping.
    missing_cols = set(artifact["feature_columns"]) - set(features.columns)
    extra_cols = set(features.columns) - set(artifact["feature_columns"])
    if missing_cols:
        print(f"[score_customers] {len(missing_cols)} training-time column(s) absent from this batch "
              f"(filled with 0): {sorted(missing_cols)}")
    if extra_cols:
        print(f"[score_customers] WARNING: {len(extra_cols)} column(s) in this batch were never seen "
              f"at training time and will be dropped: {sorted(extra_cols)}")

    aligned = align_features(features, artifact["feature_columns"])
    churn_probability = artifact["model"].predict_proba(aligned)[:, 1]

    # Reuse the standalone tier function defined above rather than
    # re-deriving the threshold logic here.
    risk_tier = [assign_risk_tier(p) for p in churn_probability]

    return pd.DataFrame(
        {
            "customerID": customer_ids.values,
            "churn_probability": churn_probability,
            "risk_tier": risk_tier,
        }
    )

## Getting 20 genuinely held-out customers

"Held-out" should mean what it means in Day 3's evaluation: never seen
during training, not just any random 20 rows. Reproducing the exact same
`train_test_split` call (same features, same `random_state=42`) recovers
the real test-set indices, then pulling the raw (pre-feature-engineered)
rows for 20 of them -- and dropping `Churn`, since real scoring input
never has it.

In [4]:
raw = pd.read_csv("../data/raw/churn_data.csv")
df = build_features(raw)
X = df.drop(columns=["churn_flag"])
y = df["churn_flag"]

_, X_test, _, _ = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

held_out_ids = raw.loc[X_test.index, "customerID"].head(20)
held_out_raw = raw.loc[held_out_ids.index].drop(columns=["Churn"])  # simulate real scoring input: no label
actual_churn = raw.loc[held_out_ids.index, ["customerID", "Churn"]]  # kept aside only for our own sanity check below

held_out_raw.shape

(20, 20)

## Score them

In [5]:
scores = score_customers(held_out_raw)
scores

,customerID,churn_probability,risk_tier
0,4376-KFVRS,0.070927,low
1,2754-SDJRD,0.976385,high
2,9917-KWRBE,0.325637,low
3,0365-GXEZS,0.728677,high
4,9385-NXKDA,0.000542,low
5,4686-UXDML,0.847758,high
6,2227-JRSJX,0.873445,high
7,4830-FAXFM,0.075192,low
8,1830-IPXVJ,0.005835,low
9,4690-LLKUA,0.718518,high


## Sanity check against actual outcome

Not a formal evaluation -- that's already done in
`notebooks/03_train_model.ipynb` on the full test set. Just eyeballing
whether the risk tiers look reasonable against what actually happened for
these 20 specific customers.

In [6]:
comparison = scores.merge(actual_churn, on="customerID").sort_values("churn_probability", ascending=False)
comparison

,customerID,churn_probability,risk_tier,Churn
1,2754-SDJRD,0.976385,high,No
6,2227-JRSJX,0.873445,high,No
5,4686-UXDML,0.847758,high,No
10,5804-HYIEZ,0.800646,high,No
3,0365-GXEZS,0.728677,high,No
9,4690-LLKUA,0.718518,high,Yes
13,4706-AXVKM,0.702431,high,Yes
17,1379-FRVEB,0.651988,medium,No
18,4753-PADAS,0.511426,medium,No
2,9917-KWRBE,0.325637,low,No


## Making training/serving skew visible: scoring a single customer

Day 3's `app/services/model.py` docstring already found that a small
batch produces fewer one-hot columns than the full training set --
`build_features()`'s `pd.get_dummies()` only emits columns for categories
actually present in whatever it's given. Scoring a single customer here to
check that concretely surfaced something worse than a merely-missing
column, though: `pd.get_dummies(..., drop_first=True)` on a batch where a
column has only *one* value present (guaranteed for a single row) collapses
to **zero** dummy columns for it, and `align_features()`'s fill-with-0 then
silently assumes that means "this customer is the baseline category" --
which is only true half the time. For a customer who is actually `Male`,
`Partner=Yes`, `Dependents=Yes`, and on `Fiber optic`, scoring them alone
originally produced `gender_Male=0` and the other three flipped the same
way -- not an approximation, the *wrong* customer.

That's the training/serving skew this task is actually about, found by
testing the thing the task asked to test, not the version of it I
originally planned to demonstrate. Fixed it at the root, in
`app/services/feature_pipeline.py`: `ONEHOT_COLUMNS` are now cast to a
`pd.Categorical` with a fixed, hardcoded set of known levels (taken from
the full raw dataset) before `get_dummies()` runs, so it always emits the
complete column set regardless of how many categories happen to appear in
a given batch -- a batch of one customer or twenty behaves identically.
Verified this doesn't require retraining: rerunning `build_features()` on
the full training set produces byte-identical columns and values to
before the fix, so the persisted `models/churn_xgb_v1.pkl` is still valid
-- the bug only ever affected small/low-diversity scoring batches, never
training.

In [7]:
single_customer = held_out_raw.head(1)
single_score = score_customers(single_customer)
single_score

,customerID,churn_probability,risk_tier
0,4376-KFVRS,0.070927,low


### Verifying the fix

Re-running the single-customer score below now matches this same
customer's probability when scored as part of the 20-row batch above
(previously: `0.0737` alone vs. `0.0709` in-batch -- now identical), and
the one-hot values themselves are correct rather than defaulted to the
baseline category.

In [8]:
from app.services.feature_pipeline import build_features as _build_features_check

single_feat = _build_features_check(single_customer)
print("gender_Male (single-row, post-fix):", bool(single_feat["gender_Male"].iloc[0]))
print("Actual gender for this customer:", raw.loc[raw["customerID"] == single_customer["customerID"].iloc[0], "gender"].iloc[0])

batch_row = scores[scores["customerID"] == single_customer["customerID"].iloc[0]]
print("\nProbability scored alone:      ", single_score["churn_probability"].iloc[0])
print("Probability scored in 20-batch:", batch_row["churn_probability"].iloc[0])
print("Match:", abs(single_score["churn_probability"].iloc[0] - batch_row["churn_probability"].iloc[0]) < 1e-5)

gender_Male (single-row, post-fix): True
Actual gender for this customer: Male

Probability scored alone:       0.07092749
Probability scored in 20-batch: 0.07092749
Match: True


## Mapping tiers to retention actions

Base mapping per the plan: high -> proactive offer/discount email,
medium -> in-app/SMS prompt, low -> no action.

One refinement, given how strongly `contract_risk` predicts churn
(Day 3's top feature by a wide margin): a high-risk month-to-month
customer and a high-risk customer already on a one/two-year contract
aren't the same situation, so they shouldn't get the same action.

- **High risk + month-to-month**: the standard play -- a discount offer
  paired with a contract-upgrade incentive, since moving them onto a
  longer contract doesn't just retain them now, it structurally lowers
  their future risk (that's what `contract_risk` is measuring).
- **High risk + one/two-year contract**: this combination is unusual --
  long-contract customers are rarely high risk at all (locking in reduces
  churn probability directly), so one who still shows up as high risk is
  an outlier the templated discount-email play wasn't designed for.
  Routing this to a personal outreach / account-manager review, rather
  than an automated email, is the more defensible default until there's
  evidence for what's actually driving their risk.
- **Medium risk**: in-app/SMS prompt, regardless of contract type -- the
  plan didn't ask for a contract split here, and adding one would be
  exactly the kind of unnecessary complexity Day 2's "don't over-engineer"
  lesson already covered.

In [9]:
def assign_retention_action(risk_tier: str, contract: str) -> str:
    """Map a risk tier (and, for high risk, contract type) to a retention action."""
    if risk_tier == "high":
        if contract == "Month-to-month":
            return "Proactive retention email: discount + contract-upgrade incentive"
        return "Escalate to account manager (atypical: high risk despite long-term contract)"
    if risk_tier == "medium":
        return "In-app / SMS engagement prompt"
    return "No action"


# Prove the contract-split branch actually fires, as a plain function call
# -- no need to find a real example in the data to test pure logic like this.
assert assign_retention_action("high", "Month-to-month") == "Proactive retention email: discount + contract-upgrade incentive"
assert assign_retention_action("high", "Two year") == "Escalate to account manager (atypical: high risk despite long-term contract)"
assert assign_retention_action("high", "One year") == "Escalate to account manager (atypical: high risk despite long-term contract)"
assert assign_retention_action("medium", "Month-to-month") == "In-app / SMS engagement prompt"
assert assign_retention_action("medium", "Two year") == "In-app / SMS engagement prompt"  # no contract split at medium
assert assign_retention_action("low", "Month-to-month") == "No action"
print("assign_retention_action: all checks passed")

assign_retention_action: all checks passed


### Wiring the rules layer onto real scores

`apply_retention_rules()` is the rules-layer step itself: it takes
`score_customers()`'s output (the ML layer) plus `Contract` from the raw
input (the one piece of business context the action rule needs that
`score_customers()` doesn't return), and produces the operational
decision. Keeping this as input -> output on plain DataFrames, not
reaching back into `score_customers()` or the model, is what keeps the
two layers genuinely separable -- a marketing team should be able to
change what "high risk" *does* without anyone retraining a model.

In [10]:
def apply_retention_rules(scores: pd.DataFrame, raw: pd.DataFrame) -> pd.DataFrame:
    """Add a retention_action column to score_customers() output."""
    with_contract = scores.merge(raw[["customerID", "Contract"]], on="customerID", how="left")
    with_contract["retention_action"] = with_contract.apply(
        lambda row: assign_retention_action(row["risk_tier"], row["Contract"]), axis=1
    )
    return with_contract.drop(columns=["Contract"])


actions = apply_retention_rules(scores, raw)
actions.sort_values("churn_probability", ascending=False)

,customerID,churn_probability,risk_tier,retention_action
1,2754-SDJRD,0.976385,high,Proactive retention email: discount + contract...
6,2227-JRSJX,0.873445,high,Proactive retention email: discount + contract...
5,4686-UXDML,0.847758,high,Proactive retention email: discount + contract...
10,5804-HYIEZ,0.800646,high,Proactive retention email: discount + contract...
3,0365-GXEZS,0.728677,high,Proactive retention email: discount + contract...
9,4690-LLKUA,0.718518,high,Proactive retention email: discount + contract...
13,4706-AXVKM,0.702431,high,Proactive retention email: discount + contract...
17,1379-FRVEB,0.651988,medium,In-app / SMS engagement prompt
18,4753-PADAS,0.511426,medium,In-app / SMS engagement prompt
2,9917-KWRBE,0.325637,low,No action


None of these 20 held-out customers happen to be both high-risk and on a
long-term contract -- expected, given `contract_risk` is the model's
dominant feature, that combination is rare by construction. The unit
tests on `assign_retention_action` above are what actually prove that
branch works, rather than hoping a real example turns up in a small
sample.

## Notes / next steps

- `score_customers(df)` reuses `build_features()` (Day 2) and the
  persisted artifact from `app/services/model.py` (Day 3) exactly --
  no reimplemented feature logic.
- `risk_tier` uses `HIGH_RISK_THRESHOLD`/`MEDIUM_RISK_THRESHOLD` from
  `.env` (Day 1) -- their first real use in the project.
- Tested on 20 genuinely held-out customers (same `train_test_split`,
  `random_state=42` as Day 3's evaluation).
- **Real bug found and fixed while testing the single-customer case**:
  `pd.get_dummies(drop_first=True)` on a low-diversity batch (guaranteed
  for a batch of one) can silently produce zero dummy columns for a
  category, and `align_features()`'s fill-0 then wrongly treated that as
  "baseline category" rather than "no diversity to tell" -- flipping real
  attributes (gender, partner status, dependents, internet type) for any
  customer scored alone or in a small, non-diverse batch. Fixed at the
  root in `app/services/feature_pipeline.py` via a fixed, hardcoded
  `CATEGORY_LEVELS` cast to `pd.Categorical` before one-hot encoding, so
  the column set is always complete regardless of batch composition.
  Verified the fix needs no retraining (identical output on the full
  training set) and does fix the single-customer case (probability now
  matches in-batch scoring exactly).
- `score_customers()` raises if handed a `Churn` column, rather than
  silently ignoring it -- scoring input for active customers should never
  have the label; if it does, something upstream is wrong and is worth
  knowing immediately, not scoring around.
- Not yet done: this logic still lives only in this notebook. A likely
  future "Folder Structure" step (matching the Day 2/Day 3 pattern) would
  move `score_customers()` into `app/services/scoring.py` once an API
  layer needs to call it -- deliberately not doing that preemptively here
  since this task asked for the notebook specifically.